In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
import os

NEWS_API_KEY = os.getenv('NEWS_API_KEY', '')

In [9]:
import requests

def fetch_news(topic: str, count: int = 5) -> list[dict]:
    if not NEWS_API_KEY:
        # 无 API key 时使用模拟数据
        return [
            {"title": f"{topic}领域重大突破", "description": f"研究人员在{topic}领域宣布取得突破性进展。", "url": "https://example.com/1", "source": {"name": "科技新闻"}},
            {"title": f"{topic}产业快速增长", "description": f"新报告显示{topic}采用率同比增长40%。", "url": "https://example.com/2", "source": {"name": "商业日报"}},
            {"title": f"专家评析{topic}面临挑战", "description": f"行业专家讨论{topic}领域面临的障碍。", "url": "https://example.com/3", "source": {"name": "科学周刊"}},
        ]

    url = f"https://newsapi.org/v2/everything?q={topic}&language=zh&pageSize={count}&sortBy=publishedAt&apiKey={NEWS_API_KEY}"
    response = requests.get(url, timeout=10)
    data = response.json()
    return data.get("articles", [])

In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

import os

model = os.getenv('MODEL_NAME', '')

def summarize_news(topic: str, articles: list[dict]) -> str:
    llm = ChatOpenAI(model=model, temperature=0, extra_body={'enable_thinking': False})

    articles_text = "\n\n".join(
        f"标题: {a['title']}\n来源: {a.get('source', {}).get('name', '未知')}\n摘要: {a.get('description', '暂无')}"
        for a in articles[:5]
    )

    print('='*200, '\n')
    print(articles_text, '\n')
    print('='*200, '\n')

    messages = [
        SystemMessage(content="你是一位新闻分析师。请生成结构化的新闻简报，包含：1) 头条新闻，2) 核心主题（3个要点），3) 值得关注的动态，4) 快速标题列表。"),
        HumanMessage(content=f"主题: {topic}\n\n新闻内容:\n{articles_text}"),
    ]

    response = llm.stream(messages)
    for chunk in response:
        print(chunk.content, end='', flush=False)

In [23]:
topic = '人工智能'
articles = fetch_news(topic=topic, count=5)
summarize_news(topic=topic, articles=articles)


标题: Meta豪擲91.7億美元 加拿大首座資料中心搶攻AI市場
来源: Yahoo Entertainment
摘要: 商傳媒｜責任編輯／綜合外電報導美國科技巨擘 Meta 週四宣布，將投入高達 91.7 億美元（約 130 億加幣），在加拿大亞伯達省的 Sturgeon County 興建其在該國的首座大型資料中心。此舉不僅是 Meta 全球第 33 座資料中心，更被視為其在全球人工智慧（AI）基礎建設佈局中的關鍵一步。 根據《KLSE Screener》報導，這座資料中心的初期容量將達到 1 Gigawatt (GW)，並具備擴展至 1.8 GW 的能力。如此龐大的投資，凸顯 Meta 對於不斷增長數據處理需求的高度重視，特別…

标题: 鉅額 AI 投資恐難即時回本 阿波羅：美國經濟面臨衰退風險
来源: Yahoo Entertainment
摘要: 圖／本報資料庫商傳媒｜責任編輯／綜合外電報導阿波羅全球管理（Apollo Global Management）首席經濟學家 Torsten Sløk 日前提出警告，指出科技巨頭在人工智慧（AI）領域的巨額投資，如果未能如期帶來預期獲利，加上非科技產業導入 AI 的進度遲緩，可能引發美國股市大幅修正，甚至導致經濟衰退。該公司預估，美國經濟在 2026 年陷入衰退的機率約為 30%。 根據報導，矽谷各大科技公司對 AI 晶片和資料中心的龐大支出，預計將從 2023 年的 2,000 億美元攀升至 2026 年的 1.…

标题: 接上 LG 螢幕竟被強塞 McAfee 廣告！Alienware、Dell 也傳出類似偷裝行為
来源: Techbang.com
摘要: 近期傳出連接 LG 螢幕竟會導致系統自動安裝軟體，並跳出惱人的 McAfee 廣告。這種現象也發生在 Dell 與 Alienware 裝置上。Reddit 論壇用戶 Mags_Smash 近日反映在未手動安裝任何防毒軟體的情況下，Windows 系統頻繁出現 McAfee 防毒軟體的彈出式廣告。經該名用戶追查系統日誌與可靠性監視器（Reliability Monitor），證實該廣告源於其新安裝的 LG 顯示器，顯示器在未經用戶授權的情況下，自動在背景下載並安裝了名為「LG Monitor App Instal…

标题: 腾讯重仓一个IPO
来源